# 6. Guardrails and terminology

Check model output and coded data for problems - PHI leaking out, claims that
their sources do not support, codes that do not exist - **without ever raising**:
every guardrail returns a result and your policy decides what a failure means.

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

## PHI leakage

The guardrail runs the same detection as the de-identifier over a piece of text
and reports categories and counts - never the value.

In [2]:
from openbtk.guardrails.phi_leakage import PHILeakageGuardrail

phi = PHILeakageGuardrail()

clean = phi.check("The patient is stable and will follow up in one week.")
leaky = phi.check("Please call the patient back at (555) 010-2345.")

print("clean:", clean.passed, "|", clean.message)
print("leaky:", leaky.passed, leaky.severity.value, "|", leaky.message)
assert "010-2345" not in leaky.model_dump_json()

clean: True | No PHI detected.
leaky: False block | Detected 1 potential PHI span(s): phone_number.


## Groundedness

Is every claim in an answer supported by its retrieved context? The default check
is a **word-overlap heuristic**: it cannot see a negation or a swapped dose, so
treat a pass as "lexically consistent", not "clinically verified". Inject your own
`is_supported` (an entailment model, an LLM judge) for a stronger check.

In [3]:
from openbtk.guardrails.groundedness import (
    GroundednessCheckInput,
    GroundednessGuardrail,
)

context = ["Assessment: type 2 diabetes mellitus, stable. Metformin continued."]

grounded = GroundednessGuardrail().check(
    GroundednessCheckInput(answer="The patient has type 2 diabetes.", context=context)
)
ungrounded = GroundednessGuardrail().check(
    GroundednessCheckInput(answer="The patient has a fractured femur.", context=context)
)
print("grounded:  ", grounded.passed, "|", grounded.message)
print("ungrounded:", ungrounded.passed, "|", ungrounded.message)

grounded:   True | All 1 claim(s) are supported.
ungrounded: False | 1 of 1 claim(s) are not supported by the retrieved context.


## Composing guardrails

`GuardrailPipeline` runs several over one payload and aggregates. With
`short_circuit=False` every one runs, for a complete report.

In [4]:
from openbtk.guardrails.pipeline import GuardrailPipeline

payload = GroundednessCheckInput(
    answer="The patient has type 2 diabetes. Call (555) 010-2345.", context=context
)
report = GuardrailPipeline(
    [PHILeakageGuardrail(), GroundednessGuardrail()], short_circuit=False
).run(payload)

print("overall passed:", report.passed)
for r in report.results:
    print(
        f"  {r.guardrail_key:32} passed={r.passed!s:5} {r.severity.value:8} {r.message}"
    )

overall passed: False
  guardrail.general.phi_leakage    passed=False block    Detected 1 potential PHI span(s): phone_number.
  guardrail.general.groundedness   passed=False block    1 of 2 claim(s) are not supported by the retrieved context.


## Terminology: what a partial vocabulary can and cannot say

OpenBTK bundles only a small **ICD-10-CM subset** (SNOMED CT and LOINC are
licensed and are never bundled). A subset can *confirm* a code it contains; it can
never say a code is *invalid*, because it might simply be missing.

In [5]:
from openbtk.core.schemas import CodeSystem
from openbtk.terminology.bundled import BundledMinimalBackend

bundled = BundledMinimalBackend()
print(bundled.resolve("E11.9", CodeSystem.ICD10CM))
print("authoritative for ICD-10-CM?", bundled.is_authoritative(CodeSystem.ICD10CM))

code='E11.9' system=<CodeSystem.ICD10CM: 'ICD10CM'> display='Type 2 diabetes mellitus without complications'
authoritative for ICD-10-CM? False


In [6]:
from openbtk.guardrails.terminology_validity import TerminologyValidityGuardrail

check = TerminologyValidityGuardrail()  # backed by the bundled subset
result = check.check([("E11.9", CodeSystem.ICD10CM), ("73211009", CodeSystem.SNOMED)])

# The SNOMED code is real; the subset just cannot confirm it - a WARNING, not a BLOCK.
print(result.severity.value, "|", result.message)
print(result.details)

warning | Could not verify 1 of 2 code(s) -- the terminology backend is unavailable, unlicensed, or does not cover them (a partial vocabulary cannot say a code is invalid).
{'unverifiable': ['SNOMED:73211009']}


With a **complete** vocabulary, absence is a real answer. Here we supply a
(tiny) vocabulary file for SNOMED, treating it as SNOMED's whole vocabulary:

In [7]:
import pathlib
import tempfile

from openbtk.terminology.local import LocalVocabBackend

vocab = pathlib.Path(tempfile.mkdtemp()) / "snomed.csv"
vocab.write_text(
    "code,system,display\n73211009,SNOMED,Diabetes mellitus\n", encoding="utf-8"
)

strict = TerminologyValidityGuardrail(terminology=LocalVocabBackend(path=str(vocab)))
for code in ("73211009", "99999999"):
    r = strict.check((code, CodeSystem.SNOMED))
    print(code, "->", r.severity.value, "|", r.message)

73211009 -> info | All 1 code(s) valid.
99999999 -> block | 1 of 1 code(s) do not exist in their declared system.


`99999999` is now a `block` (does not exist in a vocabulary we declared
complete), while a system the file has no rows for would still only be
"unverifiable".

## Limits

`UMLSRestBackend` needs your own UMLS licence and API key and is not exercised
against the live service in CI; its `map()` call in particular is unverified.